# Step 7 — Entity Extraction

## Goal

Extract structured entities from natural-language GameGuide queries.

The entity extractor will use:
- Dictionaries
- Regular expressions
- Game-name lookup
- Fuzzy matching

### Entities

- Game name
- Genre
- Tag
- Category
- Platform
- Price
- Rating
- Year
- Attribute
- Result count

In [1]:
import re
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from rapidfuzz import process, fuzz

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src import config

DATA_PATH = Path("../data/processed/games_features.parquet")
df = pd.read_parquet(DATA_PATH)

# Build popularity-prioritized name_index (most reviewed version wins)
order = df.sort_values("num_reviews_total", ascending=False, na_position="last")
name_index = {}
name_reviews = {}

for idx, name, rev in zip(order.index, order["name_clean"], order["num_reviews_total"].fillna(0)):
    if name and name not in name_index:
        name_index[name] = idx
        name_reviews[name] = rev

names_list = list(name_index.keys())
top_20k_names = names_list[:20000]  # Fast fuzzy-matching pool (Step 3 Tip)

print(f"Loaded games_features.parquet: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Indexed unique game names: {len(name_index):,} (Top pool: {len(top_20k_names):,})")

Loaded games_features.parquet: 89,550 rows × 65 columns
Indexed unique game names: 88,442 (Top pool: 20,000)


---
## 7.1 Query Preprocessing & Regular Expressions for Price, Year, and Count

We define `preprocess(q)` (from Step 6) and compile regular expressions for numeric constraints:
- **Price (`PRICE_RE`):** Matches `"under 1000"`, `"below ₹500"`, `"less than 1500"`, `"within ₹800"`, `"<= 1000"`, plus keywords `"free"` (`max_price = 0`) and `"cheap"` / `"budget"` (`max_price = 500`, justified in Step 4 EDA).
- **Year (`YEAR_AFTER_RE`, `YEAR_BEFORE_RE`, `YEAR_EXACT_RE`):** Matches `"after 2020"`, `"since 2019"`, `"before 2015"`, `"until 2018"`, and `"released in 2022"` / `"from 2021"`.
- **Result Count (`COUNT_RE`):** Matches `"top 5"`, `"10 games"`, `"show 3"`, defaulting to `config.TOP_N_DEFAULT = 5`.

In [2]:
def preprocess(q: str) -> str:
    if q is None:
        return ""
    q = str(q).lower().strip()
    q = re.sub(r"(rs\.?|rupees?|inr)\s*(\d+)", r"₹\2", q)
    q = re.sub(r"(\d+)\s*(rs\.?|rupees?|inr)", r"₹\1", q)
    q = re.sub(r"[^\w\s₹$<>.\-']", " ", q)
    return re.sub(r"\s+", " ", q).strip()

PRICE_RE = re.compile(r"(?:under|below|less than|within|<=?|max|around)\s*[₹$]?\s*(\d+)")
YEAR_AFTER_RE = re.compile(r"(?:after|since|from|post)\s*(\d{4})\b")
YEAR_BEFORE_RE = re.compile(r"(?:before|until|prior to|pre)\s*(\d{4})\b")
YEAR_EXACT_RE = re.compile(r"(?:released in|in year|games of)\s*(\d{4})\b")
COUNT_RE = re.compile(r"\b(?:top|show|give me|recommend|suggest|find)\s+(\d{1,2})\b|\b(\d{1,2})\s+(?:games|titles|recommendations)\b")

print("Compiled PRICE_RE, YEAR_AFTER_RE, YEAR_BEFORE_RE, YEAR_EXACT_RE, and COUNT_RE successfully!")

Compiled PRICE_RE, YEAR_AFTER_RE, YEAR_BEFORE_RE, YEAR_EXACT_RE, and COUNT_RE successfully!


---
## 7.2 Dictionaries for Genres, Tags, Categories, Platforms, and Attributes

We build rule-based lookup dictionaries mapped to the exact canonical values in `games_features.parquet`:
1. **`GENRE_MAP`:** Cleaned genre vocabulary plus natural synonyms (`"rpg"` $\rightarrow$ `"RPG"`, `"shooter"` $\rightarrow$ `"Action"`, `"mmo"` / `"mmorpg"` $\rightarrow$ `"Massively Multiplayer"`).
2. **`TAG_MAP`:** Top community tags (`"open world"` $\rightarrow$ `"Open World"`, `"story rich"` $\rightarrow$ `"Story Rich"`, `"souls-like"` $\rightarrow$ `"Souls-like"`, `"scary"` / `"horror"` $\rightarrow$ `"Horror"`, `"fps"` $\rightarrow$ `"FPS"`).
3. **`CATEGORY_MAP`:** `"single player"` $\rightarrow$ `"Single-player"`, `"multiplayer"` $\rightarrow$ `"Multi-player"`, `"co-op"` $\rightarrow$ `"Co-op"`.
4. **`PLATFORM_MAP`:** `"windows"` / `"pc"` $\rightarrow$ `"windows"`, `"mac"` / `"macos"` $\rightarrow$ `"mac"`, `"linux"` $\rightarrow$ `"linux"`.
5. **`ATTRIBUTE_MAP`:** Trigger keywords for Step 9 (`developer`, `publisher`, `genre`, `release_date`, `platform`, `price`, `rating`, `languages`).

In [3]:
# 1. Genre Dictionary (maps user phrase -> canonical genre in df['genres'])
GENRE_MAP = {
    "action": "Action",
    "rpg": "RPG",
    "role playing": "RPG",
    "role-playing": "RPG",
    "rpgs": "RPG",
    "adventure": "Adventure",
    "strategy": "Strategy",
    "strategic": "Strategy",
    "simulation": "Simulation",
    "simulator": "Simulation",
    "indie": "Indie",
    "casual": "Casual",
    "racing": "Racing",
    "sports": "Sports",
    "sport": "Sports",
    "massively multiplayer": "Massively Multiplayer",
    "mmorpg": "Massively Multiplayer",
    "mmo": "Massively Multiplayer",
    "free to play": "Free To Play",
    "early access": "Early Access",
    "shooter": "Action",
    "fps": "Action",
}

# 2. Tag Dictionary (maps user phrase -> canonical tag in df['tags'])
TAG_MAP = {
    "open world": "Open World",
    "open-world": "Open World",
    "story rich": "Story Rich",
    "story-rich": "Story Rich",
    "souls-like": "Souls-like",
    "soulslike": "Souls-like",
    "souls like": "Souls-like",
    "roguelike": "Rogue-like",
    "rogue-like": "Rogue-like",
    "roguelite": "Rogue-lite",
    "rogue-lite": "Rogue-lite",
    "pixel graphics": "Pixel Graphics",
    "pixel art": "Pixel Graphics",
    "sci-fi": "Sci-fi",
    "scifi": "Sci-fi",
    "science fiction": "Sci-fi",
    "horror": "Horror",
    "scary": "Horror",
    "psychological horror": "Psychological Horror",
    "survival": "Survival",
    "first-person": "First-Person",
    "first person": "First-Person",
    "fps": "FPS",
    "shooter": "Shooter",
    "third-person": "Third Person",
    "third person": "Third Person",
    "puzzle": "Puzzle",
    "platformer": "Platformer",
    "2d platformer": "2D Platformer",
    "metroidvania": "Metroidvania",
    "hack and slash": "Hack and Slash",
    "turn-based": "Turn-Based",
    "turn based": "Turn-Based",
    "tactical": "Tactical",
    "stealth": "Stealth",
    "sandbox": "Sandbox",
    "building": "Building",
    "base building": "Base Building",
    "crafting": "Crafting",
    "zombie": "Zombies",
    "zombies": "Zombies",
    "cyberpunk": "Cyberpunk",
    "dark fantasy": "Dark Fantasy",
    "fantasy": "Fantasy",
    "anime": "Anime",
    "relaxing": "Relaxing",
    "cozy": "Relaxing",
    "atmospheric": "Atmospheric",
    "visual novel": "Visual Novel",
    "point & click": "Point & Click",
    "point and click": "Point & Click",
    "VR": "VR",
    "vr": "VR",
    "2d": "2D",
    "3d": "3D",
    "difficult": "Difficult",
    "hard": "Difficult",
    "mystery": "Mystery",
    "detective": "Detective",
    "comedy": "Comedy",
    "funny": "Funny",
    "space": "Space",
    "arcade": "Arcade",
    "retro": "Retro",
}

# 3. Category Dictionary (maps user phrase -> canonical category in df['categories'])
CATEGORY_MAP = {
    "single player": "Single-player",
    "single-player": "Single-player",
    "singleplayer": "Single-player",
    "solo": "Single-player",
    "multiplayer": "Multi-player",
    "multi-player": "Multi-player",
    "pvp": "PvP",
    "online pvp": "Online PvP",
    "co-op": "Co-op",
    "coop": "Co-op",
    "cooperative": "Co-op",
    "online co-op": "Online Co-op",
    "split screen": "Shared/Split Screen",
    "controller": "Full controller support",
}

# 4. Platform Dictionary
PLATFORM_MAP = {
    "windows": "windows",
    "pc": "windows",
    "win": "windows",
    "mac": "mac",
    "macos": "mac",
    "apple": "mac",
    "linux": "linux",
    "ubuntu": "linux",
    "steampar": "linux",
}

# 5. Attribute Trigger Map (Step 7 & Step 9)
ATTRIBUTE_MAP = {
    "developer": ["who developed", "who made", "who created", "developer of", "developer behind", "developed by", "which studio", "studio behind"],
    "publisher": ["who published", "publisher of", "published by", "which company published", "publishing company"],
    "release_date": ["when was", "release date", "when did", "what year was", "which year did", "come out", "released on"],
    "price": ["how much does", "how much is", "what is the price", "price of", "cost of", "does it cost", "cost in rupees", "is it free"],
    "rating": ["what is the rating", "steam rating", "how good are the reviews", "review score", "positive rating", "reviews of", "reviews for"],
    "platform": ["available on", "playable on", "run on", "work on", "supported on", "platforms for", "can i play"],
    "languages": ["what languages", "which languages", "supported languages", "language support", "subtitles"],
    "genre": ["what genre is", "what genre does", "what are the genres", "type of game is"],
}

print(f"Loaded {len(GENRE_MAP)} genre terms, {len(TAG_MAP)} tag terms, {len(CATEGORY_MAP)} category terms, {len(PLATFORM_MAP)} platform terms, and {len(ATTRIBUTE_MAP)} attribute types.")

Loaded 22 genre terms, 64 tag terms, 14 category terms, 9 platform terms, and 8 attribute types.


---
## 7.3 Game Entity Extraction & Edge-Case Handling

We implement `extract_game(q)` with explicit safeguards for the three edge cases documented in Step 7 (Page 12):
1. **Common-word game names (`Portal`, `Control`, `Inside`, `Journey`, `Rust`, `Doom`):** Accepted when the query has game-lookup structure (e.g. `"tell me about Portal"`, `"who developed Control"`, `"games similar to Inside"`), while generic filter words (`"open world"`, `"windows"`, `"action"`, `"rpg"`, `"free"`, `"best"`) are blocked from falsely matching obscure 1-review Steam titles on search queries.
2. **Overlapping game names:** Longest n-gram window (`6` words down to `1`) wins, and ties break by highest `num_reviews_total`, so `"The Witcher 3: Wild Hunt"` beats `"Witcher"` and `"Portal with RTX"` beats `"Portal"`.
3. **Genre/tag words inside game names (e.g., `"Horror"` or `"RPG"` inside a title):** The game entity is resolved **first**, and its matched span is excluded from the query before scanning for genres, tags, and categories.

In [4]:
# Explicit trigger phrases indicating the user is referring to a specific game
GAME_Triggers_Sorted = sorted([
    "what games are similar to", "recommend games similar to", "give me recommendations similar to",
    "looking for something similar to", "looking for games similar to", "can you find me a game similar to",
    "can you suggest games like", "what other games are like", "show me games that resemble",
    "recommend some games with the same vibe as", "what are some good alternatives to",
    "which games are most similar to", "show me titles similar to", "suggest titles comparable to",
    "what should i play if i liked", "are there any games like", "anything on steam like",
    "games with gameplay similar to", "games similar to", "what games are like",
    "something like", "games that feel like", "games that play like", "alternatives to",
    "similar to", "games like", "a game like", "same vibe as", "resemble", "comparable to",
    "give me the summary and details of", "can you tell me what", "could you explain what",
    "give me a quick rundown of", "show me information about", "give me information regarding",
    "i want some details on", "give me details of", "give me an overview of",
    "tell me everything about", "tell me more about", "i want to know about",
    "details about the game", "provide information on", "brief description of",
    "describe the game", "share details of", "tell me about", "what is the game",
    "can you describe", "show details for", "can i get some info on", "game info for",
    "info about", "info on", "summary of", "explain", "describe",
    "who is the developer behind", "who is the developer of", "which company published",
    "which studio created", "who is the publisher of", "what is the release date of",
    "what is the steam rating of", "how good are the reviews for", "which languages are supported in",
    "what languages does", "what is the price of", "what is the rating of",
    "what year was", "which year did", "when was", "when did", "who developed",
    "who published", "who created", "who made", "what genre is", "what genre does",
    "what are the genres of", "how much does", "how much is", "can i play",
    "is", "does"
], key=len, reverse=True)

# Trailing phrases to strip after trigger removal (for attribute/info questions)
TRAILING_NOISE = [
    "cost in rupees", "cost on steam", "cost", "on steam", "released", "come out",
    "available on windows", "available on mac", "available on macos", "available on linux", "available on pc",
    "playable on windows", "playable on mac", "playable on linux", "playable on pc",
    "work on windows", "work on mac", "work on linux", "work on pc",
    "run on windows", "run on mac", "run on linux", "run on pc",
    "on windows", "on mac", "on macos", "on linux", "on pc",
    "belong to", "all about", "about", "support english subtitles", "support",
    "free to play", "please", "right now", "for me"
]

# Filter/query vocabulary that must NEVER be mistaken for a standalone game title in search queries
FILTER_STOP_PHRASES = (
    set(GENRE_MAP.keys())
    | set(TAG_MAP.keys())
    | set(CATEGORY_MAP.keys())
    | set(PLATFORM_MAP.keys())
    | {
        "game", "games", "title", "titles", "steam", "recommend", "recommendations",
        "suggest", "suggestions", "show", "find", "search", "looking", "looking for",
        "want", "need", "play", "playing", "some", "any", "good", "best", "top",
        "rated", "highly", "highly rated", "top rated", "good ratings", "good rating",
        "cheap", "budget", "free", "free to play", "under", "below", "less", "than",
        "within", "after", "before", "since", "until", "released", "year", "new",
        "recent", "popular", "most", "with", "for", "and", "or", "the", "a", "an",
        "in", "on", "of", "to", "me", "my", "i", "you", "can", "could", "please",
        "what", "which", "who", "when", "where", "how", "is", "are", "was", "were",
        "hello", "hi", "hey", "thanks", "thank", "thank you", "bye", "goodbye",
        "weather", "temperature", "joke", "movie", "netflix", "music", "exam", "homework"
    }
)

# Famous single-word games that are also common English words
COMMON_WORD_GAMES = {"portal", "control", "inside", "journey", "rust", "doom", "prey", "hades", "celeste", "limbo", "soma", "braid"}

def has_game_query_context(q: str) -> bool:
    """Returns True if the query contains explicit similarity, info, or attribute trigger phrasing."""
    explicit_cues = [
        "similar", " like ", "vibe as", "alternatives to", "resemble", "comparable to",
        "tell me", "details", "info ", "information", "rundown", "summary", "describe", "overview", "explain",
        "who developed", "who made", "who created", "developer", "published", "publisher",
        "when was", "when did", "what year", "which year", "release date", "come out",
        "what genre", "how much", "price of", "cost", "rating of", "steam rating", "reviews for",
        "available on", "playable on", "work on", "run on", "can i play", "languages",
        "finished ", "enjoyed ", "i loved ", "i love "
    ]
    return any(cue in q for cue in explicit_cues) or q.startswith("what is ")

def is_filter_only_span(cand: str, has_context: bool) -> bool:
    """Prevents filter terms ('open world', 'rpg', 'windows') from matching obscure Steam titles."""
    if has_context and cand in name_index and name_reviews.get(cand, 0) >= 1000 and cand not in GENRE_MAP and cand not in PLATFORM_MAP:
        return False
    if cand in FILTER_STOP_PHRASES:
        return True
    tokens = cand.split()
    if all(t in FILTER_STOP_PHRASES or t.isdigit() or t.startswith("₹") for t in tokens):
        # Allow famous common-word games ONLY when explicit game context is present
        if has_context and cand in COMMON_WORD_GAMES:
            return False
        return True
    return False

def extract_game_with_span(q: str, cutoff: int = config.FUZZY_CUTOFF):
    """
    Extracts (game_idx, matched_span, match_method, score) from preprocessed query q.
    """
    q_norm = q.replace("-", " ").replace("'", " ")
    q_norm = re.sub(r"\s+", " ", q_norm).strip()
    has_context = has_game_query_context(q)

    # Pure search/recommendation queries without game-reference triggers should not match random titles
    is_pure_search = (
        any(w in q for w in ["recommend", "suggest", "show me", "find ", "i want ", "looking for", "games released", "best ", "top ", "budget", "cheap", "free "])
        and not has_context
    )
    if is_pure_search:
        return None, "", "none", 0

    # Step A: If query has explicit trigger phrases, isolate the candidate game span first
    stripped = q_norm
    trigger_matched = False
    for trig in GAME_Triggers_Sorted:
        trig_norm = trig.replace("-", " ")
        if trig_norm in stripped:
            # Special check for short triggers "is" / "does" -> only strip if at start of query
            if trig_norm in ("is", "does"):
                if stripped.startswith(trig_norm + " "):
                    stripped = stripped[len(trig_norm):].strip()
                    trigger_matched = True
            else:
                parts = stripped.split(trig_norm, 1)
                # Keep the side that likely contains the game title
                after_part = parts[1].strip()
                before_part = parts[0].strip()
                stripped = after_part if len(after_part) >= 2 else before_part
                trigger_matched = True
                break

    if trigger_matched:
        for noise in sorted(TRAILING_NOISE, key=len, reverse=True):
            if stripped.endswith(" " + noise):
                stripped = stripped[: -len(noise)].strip()
        # Check if isolated span is an exact match in name_index
        if stripped in name_index and not is_filter_only_span(stripped, True):
            return name_index[stripped], stripped, "exact_isolated", 100

    # Step B: Slide n-gram windows (6 words down to 1) over the query
    tokens = q_norm.split()
    for n in range(min(6, len(tokens)), 0, -1):
        candidates = []
        for i in range(len(tokens) - n + 1):
            cand = " ".join(tokens[i : i + n])
            if len(cand) <= 2:
                continue
            if is_filter_only_span(cand, has_context):
                continue
            if cand in name_index:
                rev = name_reviews.get(cand, 0)
                # For 1-word matches without trigger context, require a strong popularity floor
                if n == 1 and not has_context and rev < 5000:
                    continue
                if n == 1 and has_context and cand not in COMMON_WORD_GAMES and rev < 200:
                    continue
                if n == 2 and not has_context and rev < 1000:
                    continue
                candidates.append((cand, name_index[cand], rev))
        if candidates:
            # Prefer most popular match among longest n-grams
            candidates.sort(key=lambda x: x[2], reverse=True)
            best_cand, best_idx, _ = candidates[0]
            return best_idx, best_cand, f"{n}-gram", 100

    # Step C: Fallback to RapidFuzz on trigger-stripped span (only when query has game context or is short title lookup)
    fuzzy_target = stripped if trigger_matched else q_norm
    if fuzzy_target and not is_filter_only_span(fuzzy_target, has_context) and (has_context or len(tokens) <= 4):
        # Search top 20,000 most reviewed games first (Step 3 Tip)
        match = process.extractOne(fuzzy_target, top_20k_names, scorer=fuzz.WRatio, score_cutoff=cutoff)
        if not match:
            match = process.extractOne(fuzzy_target, names_list, scorer=fuzz.WRatio, score_cutoff=cutoff)
        if match:
            matched_name, score, _ = match
            if not is_filter_only_span(matched_name, has_context):
                return name_index[matched_name], fuzzy_target, "fuzzy", round(score, 1)

    return None, "", "none", 0

def extract_game(q: str, cutoff: int = config.FUZZY_CUTOFF):
    idx, _, _, _ = extract_game_with_span(preprocess(q), cutoff=cutoff)
    return idx

print("Defined extract_game_with_span() and extract_game() successfully!")

Defined extract_game_with_span() and extract_game() successfully!


---
## 7.4 Complete Entity Extractor (`extract_entities`)

`extract_entities(query)` combines game-name resolution, span exclusion, dictionary lookups, and regex extraction into one structured dictionary:
```python
{
    "game": game_idx,          # DataFrame index or None
    "game_name": game_title,   # Canonical game name or None
    "genres": [...],           # Canonical genres list
    "tags": [...],             # Canonical tags list
    "categories": [...],       # Canonical categories list
    "platform": ...,           # 'windows' | 'mac' | 'linux' | None
    "max_price": ...,          # int (INR) or 0 for free or None
    "is_free": ...,            # bool
    "min_rating": ...,         # 80 when quality keywords are present, else None
    "year_after": ...,         # int year or None
    "year_before": ...,        # int year or None
    "attribute": ...,          # 'developer' | 'publisher' | 'release_date' | 'price' | 'rating' | 'platform' | 'languages' | 'genre' | None
    "top_n": ...               # int (default 5)
}
```

In [5]:
def extract_entities(raw_query: str) -> dict:
    q = preprocess(raw_query)

    # 1. Resolve game entity FIRST
    game_idx, matched_span, match_method, match_score = extract_game_with_span(q)
    game_name = df.loc[game_idx, "name"] if game_idx is not None else None

    # Exclude matched game span from query before extracting genres/tags/categories (Edge Case 3)
    q_remaining = q
    if matched_span:
        q_remaining = q_remaining.replace(matched_span, " ")
        # Also remove hyphenated form if applicable
        q_remaining = re.sub(re.escape(matched_span.replace(" ", "-")), " ", q_remaining)

    # 2. Extract Attribute (for specific_game_info queries)
    attribute = None
    for attr_name, triggers in ATTRIBUTE_MAP.items():
        if any(t in q for t in triggers):
            attribute = attr_name
            break

    # 3. Extract Platform
    platform = None
    for kw, plat_col in sorted(PLATFORM_MAP.items(), key=lambda x: len(x[0]), reverse=True):
        if re.search(rf"\b{re.escape(kw)}\b", q_remaining):
            platform = plat_col
            break

    # 4. Extract Price & Free/Cheap flags
    max_price = None
    is_free = False
    if m := PRICE_RE.search(q_remaining):
        max_price = int(m.group(1))
    elif re.search(r"\bfree\b", q_remaining) and attribute != "price":
        max_price = 0
        is_free = True
    elif any(w in q_remaining for w in ["cheap", "budget", "affordable", "inexpensive"]):
        max_price = 500  # Step 4 EDA budget threshold (₹500)

    # 5. Extract Year constraints
    year_after = None
    year_before = None
    if m := YEAR_AFTER_RE.search(q_remaining):
        year_after = int(m.group(1))
    if m := YEAR_BEFORE_RE.search(q_remaining):
        year_before = int(m.group(1))
    if m := YEAR_EXACT_RE.search(q_remaining):
        y = int(m.group(1))
        year_after = y - 1
        year_before = y + 1

    # 6. Extract Rating floor
    min_rating = None
    if any(w in q_remaining for w in ["highly rated", "top rated", "best", "good rating", "good ratings", "great reviews", "acclaimed"]):
        min_rating = config.HIGH_RATING

    # 7. Extract Result Count (top_n)
    top_n = config.TOP_N_DEFAULT
    if m := COUNT_RE.search(q_remaining):
        val = m.group(1) or m.group(2)
        if val:
            top_n = max(1, min(20, int(val)))

    # 8. Extract Genres, Tags, and Categories from remaining query text (unless asking "what genre is X")
    genres = []
    tags = []
    categories = []

    if attribute != "genre":
        for kw, canon_genre in sorted(GENRE_MAP.items(), key=lambda x: len(x[0]), reverse=True):
            if re.search(rf"(?<!\w){re.escape(kw)}(?!\w)", q_remaining):
                if canon_genre not in genres:
                    genres.append(canon_genre)

        for kw, canon_tag in sorted(TAG_MAP.items(), key=lambda x: len(x[0]), reverse=True):
            if re.search(rf"(?<!\w){re.escape(kw)}(?!\w)", q_remaining):
                if canon_tag not in tags:
                    tags.append(canon_tag)

        for kw, canon_cat in sorted(CATEGORY_MAP.items(), key=lambda x: len(x[0]), reverse=True):
            if re.search(rf"(?<!\w){re.escape(kw)}(?!\w)", q_remaining):
                if canon_cat not in categories:
                    categories.append(canon_cat)

    return {
        "game": game_idx,
        "game_name": game_name,
        "genres": genres,
        "tags": tags,
        "categories": categories,
        "platform": platform,
        "max_price": max_price,
        "is_free": is_free,
        "min_rating": min_rating,
        "year_after": year_after,
        "year_before": year_before,
        "attribute": attribute,
        "top_n": top_n,
    }

# Quick verification on the Step 7 Exit Check query from Page 12
exit_query = "open world rpg for windows under ₹1000"
exit_result = extract_entities(exit_query)
print(f"Step 7 Exit Check Query: '{exit_query}'")
for k, v in exit_result.items():
    print(f"  {k:12s}: {v}")

assert exit_result["genres"] == ["RPG"], f"Expected ['RPG'], got {exit_result['genres']}"
assert exit_result["tags"] == ["Open World"], f"Expected ['Open World'], got {exit_result['tags']}"
assert exit_result["platform"] == "windows", f"Expected 'windows', got {exit_result['platform']}"
assert exit_result["max_price"] == 1000, f"Expected 1000, got {exit_result['max_price']}"
assert exit_result["game"] is None, f"Expected None for game, got {exit_result['game_name']}"
print("\nPrimary Exit Check Assertion PASSED!")

Step 7 Exit Check Query: 'open world rpg for windows under ₹1000'
  game        : None
  game_name   : None
  genres      : ['RPG']
  tags        : ['Open World']
  categories  : []
  platform    : windows
  max_price   : 1000
  is_free     : False
  min_rating  : None
  year_after  : None
  year_before : None
  attribute   : None
  top_n       : 5

Primary Exit Check Assertion PASSED!


---
## 7.5 Comprehensive Unit Tests on 32 Queries (All Entities & Edge Cases)

As required by the Step 7 Deliverable (*"unit tests on 30 or more queries"*), we evaluate `extract_entities()` across **32 realistic test queries** covering:
- Multi-slot search & recommendation queries (genres, tags, categories, platforms, prices, ratings, years, counts)
- Game information and similarity queries (exact, substring, and typo/fuzzy matches)
- Specific attribute queries (`developer`, `publisher`, `release_date`, `price`, `rating`, `platform`, `languages`, `genre`)
- All documented edge cases:
  - Common-word titles (`Portal`, `Control`, `Inside`, `Journey`)
  - Overlapping titles (`The Witcher 3: Wild Hunt` vs `Witcher`, `Portal with RTX` vs `Portal`)
  - Genre/tag words embedded inside game titles

In [6]:
test_suite = [
    # 1-10: Multi-slot Search & Recommendation Queries
    ("open world rpg for windows under ₹1000",
     lambda e: e["game"] is None and "RPG" in e["genres"] and "Open World" in e["tags"] and e["platform"] == "windows" and e["max_price"] == 1000),
    ("recommend some RPG games",
     lambda e: e["game"] is None and "RPG" in e["genres"] and e["top_n"] == 5),
    ("suggest a single-player game under ₹1000 with good ratings",
     lambda e: e["game"] is None and "Single-player" in e["categories"] and e["max_price"] == 1000 and e["min_rating"] == 80),
    ("show me highly rated Linux games",
     lambda e: e["game"] is None and e["platform"] == "linux" and e["min_rating"] == 80),
    ("show games released after 2020",
     lambda e: e["game"] is None and e["year_after"] == 2020),
    ("top 10 multiplayer survival games on pc under Rs. 1500",
     lambda e: e["top_n"] == 10 and "Multi-player" in e["categories"] and "Survival" in e["tags"] and e["platform"] == "windows" and e["max_price"] == 1500),
    ("find free shooter games for mac",
     lambda e: "Action" in e["genres"] and "Shooter" in e["tags"] and e["platform"] == "mac" and e["max_price"] == 0 and e["is_free"] is True),
    ("cheap horror games below 500 rupees",
     lambda e: "Horror" in e["tags"] and e["max_price"] == 500),
    ("best co-op puzzle games released before 2018",
     lambda e: "Co-op" in e["categories"] and "Puzzle" in e["tags"] and e["min_rating"] == 80 and e["year_before"] == 2018),
    ("show 3 story rich sci-fi strategy games",
     lambda e: e["top_n"] == 3 and "Strategy" in e["genres"] and "Story Rich" in e["tags"] and "Sci-fi" in e["tags"]),

    # 11-18: Game Information & Similarity Queries (Exact, Partial, Typo)
    ("Tell me about Elden Ring",
     lambda e: e["game_name"] == "ELDEN RING" and e["genres"] == []),
    ("What games are similar to Elden Ring?",
     lambda e: e["game_name"] == "ELDEN RING"),
    ("tell me about eldn ring",
     lambda e: e["game_name"] == "ELDEN RING"),
    ("games similar to witcher 3",
     lambda e: e["game_name"] == "The Witcher 3: Wild Hunt"),
    ("give me details of The Witcher 3: Wild Hunt",
     lambda e: e["game_name"] == "The Witcher 3: Wild Hunt"),
    ("what games are like Stardew Valley",
     lambda e: e["game_name"] == "Stardew Valley"),
    ("info on Counter-Strike 2",
     lambda e: e["game_name"] == "Counter-Strike 2"),
    ("recommend games similar to Hollow Knight",
     lambda e: e["game_name"] == "Hollow Knight"),

    # 19-26: Specific Game Attribute Queries (All 8 Attributes)
    ("Who developed Elden Ring?",
     lambda e: e["game_name"] == "ELDEN RING" and e["attribute"] == "developer"),
    ("Who published Cyberpunk 2077?",
     lambda e: e["game_name"] == "Cyberpunk 2077" and e["attribute"] == "publisher"),
    ("When was Terraria released?",
     lambda e: e["game_name"] == "Terraria" and e["attribute"] == "release_date"),
    ("Is Elden Ring available on Windows?",
     lambda e: e["game_name"] == "ELDEN RING" and e["attribute"] == "platform" and e["platform"] == "windows"),
    ("How much does Baldur's Gate 3 cost?",
     lambda e: e["game_name"] == "Baldur's Gate 3" and e["attribute"] == "price"),
    ("What is the rating of Hades?",
     lambda e: e["game_name"] == "Hades" and e["attribute"] == "rating"),
    ("What genre is Stardew Valley?",
     lambda e: e["game_name"] == "Stardew Valley" and e["attribute"] == "genre" and e["genres"] == []),
    ("What languages does Dota 2 support?",
     lambda e: e["game_name"] == "Dota 2" and e["attribute"] == "languages"),

    # 27-32: Edge Cases (Common-Word Titles, Overlapping Titles, Genre Words in Title)
    ("Tell me about Portal",
     lambda e: e["game_name"] == "Portal"),
    ("Tell me about Portal with RTX",
     lambda e: e["game_name"] == "Portal with RTX"),
    ("Who developed Control?",
     lambda e: e["game_name"] == "Control Ultimate Edition" and e["attribute"] == "developer"),
    ("Games similar to Inside",
     lambda e: e["game_name"] == "INSIDE"),
    ("What is Journey about?",
     lambda e: e["game_name"] == "Journey"),
    ("budget roguelike pixel graphics games",
     lambda e: e["game"] is None and e["max_price"] == 500 and "Rogue-like" in e["tags"] and "Pixel Graphics" in e["tags"]),
]

results_rows = []
passed = 0

for idx_q, (q_text, check_fn) in enumerate(test_suite, start=1):
    ent = extract_entities(q_text)
    ok = bool(check_fn(ent))
    passed += int(ok)
    results_rows.append({
        "#": idx_q,
        "Query": q_text,
        "Game": ent["game_name"],
        "Genres": ", ".join(ent["genres"]) or "-",
        "Tags": ", ".join(ent["tags"]) or "-",
        "Cat": ", ".join(ent["categories"]) or "-",
        "Plat": ent["platform"] or "-",
        "Price": ent["max_price"] if ent["max_price"] is not None else "-",
        "Rating": ent["min_rating"] or "-",
        "Year": f">{ent['year_after']}" if ent["year_after"] else (f"<{ent['year_before']}" if ent["year_before"] else "-"),
        "Attr": ent["attribute"] or "-",
        "TopN": ent["top_n"],
        "Pass": "PASS" if ok else "FAIL"
    })

results_df = pd.DataFrame(results_rows)
print("=" * 105)
print(f"GAMEGUIDE — STEP 7 ENTITY EXTRACTION UNIT TEST RESULTS: {passed}/{len(test_suite)} PASSED ({passed/len(test_suite)*100:.1f}%)")
print("=" * 105)
print(results_df.to_string(index=False))
assert passed == len(test_suite), f"Expected all {len(test_suite)} tests to pass, but {len(test_suite) - passed} failed!"
print("\nAll 32 Entity Extraction Unit Tests PASSED!")

GAMEGUIDE — STEP 7 ENTITY EXTRACTION UNIT TEST RESULTS: 32/32 PASSED (100.0%)
 #                                                      Query                     Game   Genres                       Tags           Cat    Plat Price Rating  Year         Attr  TopN Pass
 1                     open world rpg for windows under ₹1000                     None      RPG                 Open World             - windows  1000      -     -            -     5 PASS
 2                                   recommend some RPG games                     None      RPG                          -             -       -     -      -     -            -     5 PASS
 3 suggest a single-player game under ₹1000 with good ratings                     None        -                          - Single-player       -  1000     80     -            -     5 PASS
 4                           show me highly rated Linux games                     None        -                          -             -   linux     -     80     -       

# Step 7 — Entity Extraction Complete

### Completed Tasks
- Built rule-based dictionaries for **Genres (`GENRE_MAP`)**, **Tags (`TAG_MAP`)**, **Categories (`CATEGORY_MAP`)**, **Platforms (`PLATFORM_MAP`)**, and **Attributes (`ATTRIBUTE_MAP`)** aligned with the canonical values in `games_features.parquet`.
- Compiled regular expressions for **price limits (`PRICE_RE`)**, **release year filters (`YEAR_AFTER_RE`, `YEAR_BEFORE_RE`, `YEAR_EXACT_RE`)**, and **result count (`COUNT_RE`)**.
- Implemented `extract_game_with_span()` and `extract_entities()` with n-gram windowing (`6` down to `1`), popularity tie-breaking, RapidFuzz fallback on the top 20,000 games, and full edge-case handling:
  - **Common-word titles (`Portal`, `Control`, `Inside`, `Journey`)** are resolved when game-lookup context is present while filter-only search queries (`"open world rpg for windows under ₹1000"`) never trigger false game matches.
  - **Overlapping titles** resolve to the longest specific match (`"The Witcher 3: Wild Hunt"` over `"Witcher"`, `"Portal with RTX"` over `"Portal"`).
  - **Matched game spans are excluded** before scanning for genres, tags, and categories.
- Validated `extract_entities()` on **32 unit test queries** with **32/32 (`100%`) passing**.